# OT in linear ICA

## Experiment 1: The "Full Hybrid" Stress Test
### Benchmarking Robustness Against Statistical Heterogeneity

While isolating specific failure modes is scientifically useful, real-world data is rarely statistically uniform. This experiment introduces a **Full Hybrid** environment to test the limits of algorithmic robustness in high-dimensional spaces (30-40 dimensions).

#### The Challenge: A Statistical Cocktail
We challenge the algorithms to simultaneously unmix a chaotic mixture of 8 distinct statistical "flavors":
* **Super-Gaussian (Heavy Tails):** Laplace, Student-t
* **Sub-Gaussian (Bounded):** Uniform
* **Skewed / Strictly Positive:** Chi-square, Exponential
* **Discrete / Stepped:** Bernoulli, Poisson, Binomial

#### Why Hybrid Mixtures Break Traditional ICA
Parametric algorithms like **FastICA** rely on a specific contrast function (e.g., `logcosh`) to approximate non-Gaussianity. This assumes a somewhat uniform "type" of non-Gaussianity across all latent sources. In a hybrid mixture, the parametric approximations clash—FastICA’s Newton solver often becomes confused by the conflicting local curvatures of skewed vs. sub-Gaussian signals, leading to erratic convergence and high **Amari Error**.

#### The Geometric Solution (OT-ICA)
**OT-ICA** utilizes the Wasserstein-2 distance to measure the global geometric discrepancy between the data and a Gaussian target. Theoretically, this metric is agnostic to the "type" of non-Gaussianity (skewed, heavy-tailed, or discrete). This experiment benchmarks whether this geometric approach can maintain stability where local density approximations fail.

In [ ]:
import numpy as np
import torch
import pandas as pd
import time
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import FastICA
from sklearn.exceptions import ConvergenceWarning
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import gc

from wasserstein_ica import WassersteinICA

In [ ]:
# ==========================================
# 1. Global Thesis Configuration & Helpers
# ==========================================
def set_thesis_theme():
    thesis_colors = ['#0173B2', '#DE8F05', '#029E73', '#D55E00', '#CC78BC', '#CA9161']
    plt.rcParams.update({
        'figure.figsize': (24, 10), # Doubled figure size
        'figure.dpi': 300,
        'axes.prop_cycle': plt.cycler(color=thesis_colors),
        'axes.grid': True,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'axes.axisbelow': True,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'font.size': 24,           # Doubled base font size
        'axes.titlesize': 28,      # Doubled title size
        'axes.labelsize': 24,      # Doubled label size
        'legend.frameon': True,
        'legend.fontsize': 22,     # Doubled legend font size
        'lines.linewidth': 4.5     # Thicker lines for huge plots
    })

set_thesis_theme()

In [ ]:
def amari_error(W, A):
    if W is None or np.any(np.isnan(W)): return np.nan
    P = np.abs(np.dot(W, A))
    n = P.shape[0]
    row_sum = np.sum(P / np.max(P, axis=1, keepdims=True), axis=1) - 1.0
    col_sum = np.sum(P / np.max(P, axis=0, keepdims=True), axis=0) - 1.0
    return (np.sum(row_sum) + np.sum(col_sum)) / (2 * n)

In [ ]:
# ==========================================
# 2. Unified Data Generators
# ==========================================
def generate_mixture(n_dim, n_samples, config_type, param=None, seed=None):
    if seed is not None: np.random.seed(seed)
    
    # Standard Generators
    def gen_laplace(): return np.random.laplace(0, 1/np.sqrt(2), size=n_samples)
    def gen_bernoulli(): return np.random.choice([-1.0, 1.0], size=n_samples)
    def gen_uniform(): return np.random.uniform(-np.sqrt(3), np.sqrt(3), size=n_samples)
    def gen_student_t(): s = np.random.standard_t(df=3, size=n_samples); return s / np.std(s)
    
    # Parameterized Discrete Generators
    def gen_poisson(lam=3.0): s = np.random.poisson(lam=lam, size=n_samples); return (s - np.mean(s)) / np.std(s)
    def gen_binomial(n=10, p=0.5): s = np.random.binomial(n=n, p=p, size=n_samples); return (s - np.mean(s)) / np.std(s)
    
    # Positive continuous
    def gen_chisquare(): s = np.random.chisquare(df=2, size=n_samples); return (s - np.mean(s)) / np.std(s)
    def gen_exponential(): s = np.random.exponential(scale=1.0, size=n_samples); return (s - np.mean(s)) / np.std(s)

    pools = {
        'Full Hybrid': [gen_laplace, gen_bernoulli, gen_uniform, gen_student_t, gen_poisson, gen_binomial, gen_chisquare, gen_exponential],
        'Continuous Only': [gen_laplace, gen_uniform, gen_student_t, gen_chisquare, gen_exponential],
        'Discrete Only': [gen_bernoulli, gen_poisson, gen_binomial],
        'Strictly Super-Gaussian': [gen_laplace, gen_student_t, gen_chisquare, gen_exponential],
        'Zero Gaussian': [gen_laplace, gen_bernoulli, gen_uniform, gen_student_t, gen_poisson, gen_binomial, gen_chisquare, gen_exponential]
    }
    
    sources = []
    
    # Mode A: Ablation Pools
    if config_type in pools:
        active_pool = pools[config_type]
        if config_type != 'Zero Gaussian':
            sources.append(np.random.normal(0, 1, size=n_samples)) # Add Gaussian
            n_to_gen = n_dim - 1
        else:
            n_to_gen = n_dim
        for _ in range(n_to_gen): sources.append(np.random.choice(active_pool)())
            
    # Mode B: Single Isolated Discrete Type
    else:
        sources.append(np.random.normal(0, 1, size=n_samples)) # Baseline Gaussian
        for _ in range(n_dim - 1):
            if config_type == 'Bernoulli': sources.append(gen_bernoulli())
            elif config_type == 'Poisson': sources.append(gen_poisson(lam=param if param else 3.0))
            elif config_type == 'Binomial': sources.append(gen_binomial(n=param if param else 10))
            
    S = np.stack(sources)
    np.random.shuffle(S) # Hide the Gaussian
    
    cond_num = 1000
    while cond_num > 100:
        A = np.random.randn(n_dim, n_dim)
        cond_num = np.linalg.cond(A)
    return A @ S, A

In [ ]:
# ==========================================
# 3. Unified Worker Function (UPDATED)
# ==========================================
def run_ica_trial(dim, trial, n_samples, config_type, method, param=None):
    torch.set_num_threads(1)
    X_np, A_true = generate_mixture(dim, n_samples, config_type, param, seed=trial)
    
    score = np.nan
    converged = False

    if method == 'FastICA':
        # --- Oracle Restarts added for FastICA ---
        best_score = np.inf
        best_conv = False
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            for r in range(50): # 50 Random initializations
                try:
                    seed_r = trial * 1000 + r
                    fast_ica = FastICA(n_components=dim, max_iter=10000, tol=1e-4, random_state=seed_r)
                    fast_ica.fit(X_np.T)
                    current_score = amari_error(fast_ica.components_, A_true)
                    
                    if current_score < best_score:
                        best_score = current_score
                        if not np.isnan(current_score) and current_score < 1.0: 
                            best_conv = True
                except Exception: 
                    pass
                    
        if not np.isinf(best_score):
            score = best_score
            converged = best_conv

    elif method == 'OT-ICA':
        try:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            print("Running on: ", device)
            X_torch = torch.tensor(X_np, dtype=torch.float32).to(device)
            ica = WassersteinICA(X_torch)
            ica.whiten()
            W_white = ica.W_white.cpu().numpy()
            
            extracted = []
            n_restarts = min(dim * 4, 150)
            for _ in range(dim):
                prev = torch.stack(extracted) if extracted else None
                w, _ = ica.optimize_wasserstein2(prev_components=prev, max_iter=200, n_restarts=n_restarts, dither_sigma=0.01)
                extracted.append(w)
                
            W_stiefel = ica.optimize_symmetric(n_components=dim, max_iter=400, lr=0.25, init_w=torch.stack(extracted), optimizer='stiefel', dither_sigma=0.01, batch_size=1024)
            score = amari_error(W_stiefel.cpu().numpy() @ W_white, A_true)
            if not np.isnan(score) and score < 1.0: converged = True
        except Exception: pass
    
    # --- Aggressive Memory Cleanup ---
    try:
        del X_np
        del A_true
        if 'X_torch' in locals():
            del X_torch
        if 'ica' in locals():
            del ica
        if 'fast_ica' in locals():
            del fast_ica
    except NameError:
        pass
        
    gc.collect() # Force the OS to reclaim the RAM immediately

    return {'Dim': dim, 'Config': config_type, 'Config_Param': param, 'Method': method, 'Error': score, 'Conv': int(converged)}

In [ ]:
def plot_results(df, title, palette, x_col='Config'):
    # 1. Prepare Amari Error Data
    df_plot = df.copy()
    df_plot['Error'] = df_plot['Error'].fillna(1.5) 

    # 2. Prepare Convergence Data
    df_conv = df[df['Method'] == 'FastICA'].groupby(['Dim', x_col, 'Method', 'Trial_Index'] if 'Trial_Index' in df.columns else [x_col, 'Method'])['Conv'].mean().reset_index()
    # If trial index is missing, we calculate raw mean over groups, but CI needs raw boolean observations
    df_conv_raw = df[df['Method'] == 'FastICA'].copy()
    df_conv_raw['Success Rate (%)'] = df_conv_raw['Conv'] * 100

    # --- Massive Font Size Controls ---
    TITLE_FS = 32   
    LABEL_FS = 28   
    TICK_FS = 24    
    LEGEND_FS = 24  
    # --------------------------------------

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))
    
    # --- Subplot 1: Amari Error (Lineplot with CI band) ---
    sns.lineplot(data=df_plot, x=x_col, y='Error', hue='Method', ax=ax1, palette=palette, marker='o', linewidth=4.5, markersize=14, errorbar='ci')
    
    ax1.set_title(f"Amari Error: {title}\n(Lower is Better)", fontsize=TITLE_FS, pad=20, fontweight='bold')
    ax1.set_xlabel(x_col, fontsize=LABEL_FS, labelpad=15)
    ax1.set_ylabel('Error', fontsize=LABEL_FS, labelpad=15)
    ax1.tick_params(axis='both', labelsize=TICK_FS)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=30, ha="right")
    ax1.legend(title='Method', fontsize=LEGEND_FS, title_fontsize=LEGEND_FS)
    ax1.set_ylim(0, 2.0) 

    # --- Subplot 2: Convergence Rate (Lineplot with CI band) ---
    fast_palette = {'FastICA': palette['FastICA']}
    sns.lineplot(data=df_conv_raw, x=x_col, y='Success Rate (%)', hue='Method', ax=ax2, palette=fast_palette, marker='o', linewidth=4.5, markersize=14, errorbar='ci')
    
    ax2.set_title(f"Mechanical Convergence Rate: {title}\n(FastICA Only)", fontsize=TITLE_FS, pad=20, fontweight='bold')
    ax2.set_xlabel(x_col, fontsize=LABEL_FS, labelpad=15)
    ax2.set_ylabel('Success Rate (%)', fontsize=LABEL_FS, labelpad=15)
    ax2.tick_params(axis='both', labelsize=TICK_FS)
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=30, ha="right")
    ax2.legend(title='Method', fontsize=LEGEND_FS, title_fontsize=LEGEND_FS)
    ax2.set_ylim(0, 105)

    plt.tight_layout()
    
    # Generate automatic safe file name for PDF
    file_name = f"{title.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '').replace('-', '_').lower()}.pdf"
    plt.savefig(file_name, format='pdf', bbox_inches='tight')
    plt.show()
    
    display(df.groupby(['Method', x_col, 'Dim'])[['Error', 'Conv']].mean().round(4))

In [ ]:
DIMENSIONS = [30, 40]
POOLS = ['Continuous Only', 'Discrete Only', 'Strictly Super-Gaussian', 'Zero Gaussian', 'Full Hybrid']
METHODS = ['FastICA', 'OT-ICA']
N_SAMPLES = 10000; N_TRIALS = 5 

print("--- Step 1: Sub-Pool Ablation Study ---")
tasks = [(dim, trial, N_SAMPLES, p, m, None) for dim in DIMENSIONS for p in POOLS for m in METHODS for trial in range(N_TRIALS)]
results_ablation = Parallel(n_jobs=8)(delayed(run_ica_trial)(*t) for t in tqdm(tasks))

df_ablation = pd.DataFrame(results_ablation)

In [ ]:
plot_results(df_ablation, "Sub-Pool Ablation", palette={'FastICA': '#0173B2', 'OT-ICA': '#D55E00'})

# Experiment 2:Discrete Failures
### Jagged Landscapes vs. Insufficient Non-Gaussianity

Previous tests show that both FastICA and OT-ICA struggle with discrete distributions like Poisson and Binomial. This experiment isolates the two potential causes for these failures:

#### 1. The "Gaussian Blur" (Data Limitation)
By the Central Limit Theorem, distributions like Poisson (high $\lambda$) or Binomial (high $n$) rapidly approach a Gaussian shape. If the Wasserstein gap is too small, the signal is fundamentally inseparable, regardless of the algorithm used.

#### 2. Geometric Jaggedness (Optimization Limitation)
Discrete data lacks a smooth probability density. This creates a "stair-stepped" optimization landscape filled with sharp local minima. We investigate whether these "jagged traps" stall gradient-based solvers even when the signal is mathematically non-Gaussian.

#### Experiment Roadmap
* **Sub-Pool Ablation:** Comparing unmixing performance on purely continuous pools versus purely discrete pools.
* **Surface Bifurcation Analysis:** Mapping the raw objective landscapes of $W_2^2$ vs. Logcosh Proxy Negentropy to visualize "Harsh" (low $\lambda/n$) vs. "Standard" discrete signals.
* **Compute Regime Stress Test:** Determining if high-restart, high-iteration optimization can "power through" jagged landscapes when a valid non-Gaussian signal exists.

In [ ]:
DIST_TYPES = ['Bernoulli', 'Poisson', 'Binomial'] 

print("--- Step 2: Isolating Discrete Failure Modes ---")
tasks_disc = [(dim, trial, N_SAMPLES, d, m, None) for dim in DIMENSIONS for d in DIST_TYPES for m in METHODS for trial in range(N_TRIALS)]
results_discrete = Parallel(n_jobs=8)(delayed(run_ica_trial)(*t) for t in tqdm(tasks_disc))

df_discrete = pd.DataFrame(results_discrete)

In [ ]:
plot_results(df_discrete, "Isolated Discrete Distributions", palette={'FastICA': '#0173B2', 'OT-ICA': '#D55E00'})

In [ ]:
import numpy as np
import torch

def get_analytical_gaussian_target(n_samples):
    p = torch.linspace(0, 1, n_samples + 2)[1:-1]
    return torch.distributions.Normal(0, 1).icdf(p)

print("--- Step 3A: Non-Gaussianity Sensitivity Analysis (W2 Gap & Logcosh) ---")
n_samples = 10000
target = get_analytical_gaussian_target(n_samples)

v_large = np.random.normal(0, 1, 100000)
C_gauss = np.mean(np.log(np.cosh(v_large)))

dists = {
    'Poisson (Standard, λ=3.0)': np.random.poisson(3.0, n_samples),
    'Poisson (Harsh, λ=0.5)': np.random.poisson(0.5, n_samples),
    'Binomial (Standard, n=10)': np.random.binomial(10, 0.5, n_samples),
    'Binomial (Harsh, n=2)': np.random.binomial(2, 0.5, n_samples),
    'Laplace (Baseline)': np.random.laplace(0, 1, n_samples)
}

print(f"{'Distribution':<30} | {'W2 Sq Distance':<15} | {'Logcosh Proxy Negentropy':<25}")
print("-" * 76)
for name, samples in dists.items():
    s = (samples - samples.mean()) / samples.std() 
    
    sorted_s, _ = torch.sort(torch.tensor(s, dtype=torch.float32))
    w2_val = torch.mean((sorted_s - target)**2).item()
    
    logcosh_val = (np.mean(np.log(np.cosh(s))) - C_gauss)**2
    
    print(f"{name:<30} | {w2_val:<15.6f} | {logcosh_val:<25.6f}")

In [ ]:
# --- Font Size Controls ---
SUPTITLE_FS = 32  
TITLE_FS = 28     
LABEL_FS = 26     
TICK_FS = 24      
LEGEND_FS = 24    
# --------------------------------------

def get_analytical_gaussian_target(n_samples, device='cpu'):
    p = torch.linspace(0, 1, n_samples + 2, device=device)[1:-1]
    return torch.distributions.Normal(0, 1).icdf(p)

def plot_surface_on_ax(ax, title, dist_name, param_name, param_val, S, target, C_gauss):
    thetas = np.linspace(0, np.pi, 250)
    w2_curve = []
    fastica_curve = []
    
    for th in thetas:
        w = np.array([np.cos(th), np.sin(th)])
        y = w @ S
        y_std = (y - y.mean()) / y.std()
        
        y_torch = torch.sort(torch.tensor(y_std, dtype=torch.float32))[0]
        w2_curve.append(torch.mean((y_torch - target)**2).item())
        
        proxy_negentropy = (np.mean(np.log(np.cosh(y_std))) - C_gauss)**2
        fastica_curve.append(proxy_negentropy)
        
    color_w2 = '#DE8F05' 
    ax.plot(thetas, w2_curve, color=color_w2, linewidth=4.5, label='OT-ICA ($W_2^2$)')
    ax.set_ylabel('W2 Distance ($W_2^2$)', color=color_w2, fontsize=LABEL_FS, labelpad=15) 
    ax.tick_params(axis='y', labelcolor=color_w2, labelsize=TICK_FS)          
    ax.tick_params(axis='x', labelsize=TICK_FS)                               
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xticks([0, np.pi/2, np.pi])
    ax.set_xticklabels(['0', '$\\pi/2$', '$\\pi$'], fontsize=TICK_FS)         
    
    ax.set_title(f"{title}: ${param_name} = {param_val}$", fontsize=TITLE_FS, pad=15, fontweight='bold') 
    
    ax_fast = ax.twinx()
    color_fast = '#0173B2' 
    ax_fast.plot(thetas, fastica_curve, color=color_fast, linewidth=4.5, linestyle='--', label='FastICA (Logcosh)')
    ax_fast.set_ylabel('Proxy Negentropy', color=color_fast, fontsize=LABEL_FS, labelpad=15) 
    ax_fast.tick_params(axis='y', labelcolor=color_fast, labelsize=TICK_FS)     
    
    return ax_fast

In [ ]:
# ==========================================
# Main execution to build the 2x2 grid
# ==========================================
n_samples = 10000
device = 'cpu' 

target_quantiles = get_analytical_gaussian_target(n_samples, device=device)

v_large = np.random.normal(0, 1, 100000)
C_gauss_static = np.mean(np.log(np.cosh(v_large)))

s_gauss_src = np.random.normal(0, 1, n_samples)

plots_spec = [
    {'title': 'Harsh Poisson', 'dist_name': 'Poisson', 'param_name': 'lambda', 'param_val': 0.5, 'pos': (0,0)},
    {'title': 'Standard Poisson', 'dist_name': 'Poisson', 'param_name': 'lambda', 'param_val': 3.0, 'pos': (0,1)},
    {'title': 'Harsh Binomial', 'dist_name': 'Binomial', 'param_name': 'n', 'param_val': 3, 'pos': (1,0)},
    {'title': 'Standard Binomial', 'dist_name': 'Binomial', 'param_name': 'n', 'param_val': 10, 'pos': (1,1)}
]

fig, axs = plt.subplots(2, 2, figsize=(24, 16), sharex=True)

axs_flat = axs.flatten()
final_handles = []

for i, spec in enumerate(plots_spec):
    ax = axs[spec['pos']]
    np.random.seed(i) 
    
    if spec['dist_name'] == 'Poisson':
        s_src = np.random.poisson(spec['param_val'], n_samples)
    else: 
        s_src = np.random.binomial(spec['param_val'], 0.5, n_samples)
        
    s_std_src = (s_src - np.mean(s_src)) / np.std(s_src)
    
    S_mixed = np.vstack([s_std_src, s_gauss_src])
    
    ax_fast_returned = plot_surface_on_ax(ax, spec['title'], spec['dist_name'], spec['param_name'], spec['param_val'], S_mixed, target_quantiles, C_gauss_static)
    
    if i == len(plots_spec) - 1:
        lines_w2, labels_w2 = ax.get_legend_handles_labels()
        lines_fast, labels_fast = ax_fast_returned.get_legend_handles_labels()
        final_handles = lines_w2 + lines_fast

for i, ax in enumerate(axs.flatten()):
    if i >= 2: 
        ax.set_xlabel('Projection Angle $\\theta$ (Radians)', fontsize=LABEL_FS, labelpad=15) 

fig.suptitle("Optimization Landscape Bifurcation: Pure Discrete Mixtures\nComparing Global Geometric Matching (OT-ICA) vs. Local Density Approximation (FastICA)", fontsize=SUPTITLE_FS, y=1.05, fontweight='bold') 
fig.legend(final_handles, ['OT-ICA ($W_2^2$ Distance)', 'FastICA (Logcosh Negentropy)'], loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.05), fontsize=LEGEND_FS, frameon=True) 
fig.tight_layout()

fig.savefig('all_optimization_surfaces.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
print("--- Step 4: Harsh Discrete Test (λ=0.5, n=3) ---")
HARSH_DIST = [('Poisson', 0.5), ('Binomial', 3)]

DIMENSIONS = [30, 40]
N_SAMPLES = 10000
METHODS = ['FastICA', 'OT-ICA']
N_TRIALS = 5

tasks_harsh = [(dim, trial, N_SAMPLES, d, m, p) for dim in DIMENSIONS for (d, p) in HARSH_DIST for m in METHODS for trial in range(N_TRIALS)]
results_harsh = Parallel(n_jobs=8)(delayed(run_ica_trial)(*t) for t in tqdm(tasks_harsh))

df_harsh = pd.DataFrame(results_harsh)

if 'Config_Param' not in df_harsh.columns:
    df_harsh['Config_Param'] = [task[5] for task in tasks_harsh]

df_harsh['Config'] = df_harsh.apply(
    lambda row: f"{row['Config']} (param={row['Config_Param']})" if pd.notna(row.get('Config_Param')) else row['Config'], 
    axis=1
)

In [ ]:
def format_label(row, x_col='Config'):
    name = str(row[x_col])
    param_val = None
    
    if '(param=' in name:
        base = name.split(' (param=')[0]
        param_val = name.split('param=')[1].replace(')', '')
    elif 'Config_Param' in row and pd.notna(row['Config_Param']):
        base = name
        param_val = str(row['Config_Param'])
    else:
        return name 
        
    if 'Poisson' in base:
        return f"{base} (λ={param_val})"
    elif 'Binomial' in base:
        try:
            n_val = int(float(param_val))
            return f"{base} (n={n_val}, p=0.5)"
        except ValueError:
            return f"{base} (n={param_val}, p=0.5)"
    elif 'Bernoulli' in base:
        return f"{base} (p={param_val})"
    else:
        return f"{base} ({param_val})"
        
df_harsh['Config'] = df_harsh.apply(lambda row: format_label(row), axis=1)

In [ ]:
plot_results(df_harsh, "Harsh Discrete Environments", palette={'FastICA': '#0173B2', 'OT-ICA': '#D55E00'}, x_col='Config')

In [ ]:
def run_ica_regime_trial(dim, trial, n_samples, config_type, method, param, compute_level):
    torch.set_num_threads(1)
    X_np, A_true = generate_mixture(dim, n_samples, config_type, param, seed=trial)
    
    score = np.nan
    converged = False

    if compute_level == 'Low Compute':
        w2_restarts = min(dim * 4, 150)
        w2_def_iter = 200
        w2_sym_iter = 400
    else: 
        w2_restarts = min(dim * 15, 600)
        w2_def_iter = 300
        w2_sym_iter = 500

    if method == 'FastICA':
        # --- Oracle Restarts added for FastICA ---
        best_score = np.inf
        best_conv = False
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            for r in range(50):
                try:
                    seed_r = trial * 1000 + r
                    fast_ica = FastICA(n_components=dim, max_iter=10000, tol=1e-4, random_state=seed_r)
                    fast_ica.fit(X_np.T)
                    current_score = amari_error(fast_ica.components_, A_true)
                    
                    if current_score < best_score:
                        best_score = current_score
                        if not np.isnan(current_score) and current_score < 1.0: 
                            best_conv = True
                except Exception: 
                    pass
                    
        if not np.isinf(best_score):
            score = best_score
            converged = best_conv

    elif method == 'OT-ICA':
        try:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            print("Running on: ", device)
            X_torch = torch.tensor(X_np, dtype=torch.float32).to(device)
            ica = WassersteinICA(X_torch)
            ica.whiten()
            W_white = ica.W_white.cpu().numpy()
            
            extracted = []
            for _ in range(dim):
                prev = torch.stack(extracted) if extracted else None
                w, _ = ica.optimize_wasserstein2(
                    prev_components=prev, 
                    max_iter=w2_def_iter, 
                    n_restarts=w2_restarts, 
                    dither_sigma=0.01
                )
                extracted.append(w)
                
            W_stiefel = ica.optimize_symmetric(
                n_components=dim, 
                max_iter=w2_sym_iter, 
                lr=0.25, 
                init_w=torch.stack(extracted), 
                optimizer='stiefel', 
                dither_sigma=0.01, 
                batch_size=1024
            )
            score = amari_error(W_stiefel.cpu().numpy() @ W_white, A_true)
            if not np.isnan(score) and score < 1.0: converged = True
        except Exception: pass

    # --- Aggressive Memory Cleanup ---
    try:
        del X_np
        del A_true
        if 'X_torch' in locals():
            del X_torch
        if 'ica' in locals():
            del ica
        if 'fast_ica' in locals():
            del fast_ica
    except NameError:
        pass
        
    gc.collect() # Force the OS to reclaim the RAM immediately

    return {'Dim': dim, 'Config': config_type, 'Config_Param': param, 'Method': method, 'Error': score, 'Conv': int(converged)}

In [ ]:
print("--- Final Stress Test: Harsh Discrete vs. Compute Regimes ---")
HARSH_DIST = [('Poisson', 0.5), ('Binomial', 3)]
DIMENSIONS = [30, 40]
COMPUTE_REGIMES = ['Low Compute', 'High Compute']
METHODS = ['FastICA', 'OT-ICA']
N_TRIALS = 5

tasks_regime = [
    (dim, trial, 10000, dist, method, param, regime) 
    for dim in DIMENSIONS 
    for (dist, param) in HARSH_DIST 
    for method in METHODS 
    for regime in COMPUTE_REGIMES 
    for trial in range(N_TRIALS)
]

results_regime = Parallel(n_jobs=8)(delayed(run_ica_regime_trial)(*t) for t in tqdm(tasks_regime))
df_regime = pd.DataFrame(results_regime)
df_regime['Config'] = df_regime.apply(lambda row: format_label(row), axis=1)

In [ ]:
plot_results(df_regime, "Harsh Discrete (Regime Scaling)", palette={'FastICA': '#0173B2', 'OT-ICA': '#D55E00'}, x_col='Config')